In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e2/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e2/train.csv
/kaggle/input/competitions/playground-series-s6e2/test.csv


In [2]:
df=pd.read_csv("/kaggle/input/competitions/playground-series-s6e2/train.csv")
df_test=pd.read_csv("/kaggle/input/competitions/playground-series-s6e2/test.csv")
df.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [3]:
len(df.columns)

15

In [4]:

df["Heart Disease"]=(df["Heart Disease"]=="Presence").astype(int)
columns=list(df.columns)
x=df[[column for column in columns if column != 'Heart Disease']]
y=df["Heart Disease"]

x_test_df=df_test

In [5]:
#feature selection 
#using PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(x)
pca = PCA(n_components=0.90) #keeping 90% variance
X_pca = pca.fit_transform(X_scaled)

#for test set
x_test_scaled = scaler.transform(x_test_df)
x_test_pca = pca.transform(x_test_scaled)

print(X_pca.shape)

print(pca.explained_variance_ratio_)
print(sum(pca.explained_variance_ratio_))

(630000, 12)
[0.21117111 0.07156897 0.07147027 0.07120108 0.07076036 0.06904979
 0.06617339 0.06491583 0.05946903 0.05526731 0.05452604 0.05204624]
0.9176194206424763


In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_pca,
    y,
    test_size=0.2,
    random_state=42
)

In [7]:
#Applying Neural Networks for classfication

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import linear, relu, sigmoid
tf.random.set_seed(1234) # for consistent results
model = Sequential(
    [               
        ### START CODE HERE ### 
        tf.keras.Input(shape=(12,)), #necessray to give input dimension also
        Dense(units=15,activation="relu"),
        Dense(units=8,activation="relu"),
        Dense(units=1,activation="sigmoid")

        ### END CODE HERE ### 
    ], name = "my_model" 
)

2026-05-18 12:11:54.492044: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779106314.693583      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779106314.753592      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779106315.228915      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779106315.228974      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779106315.228977      16 computation_placer.cc:177] computation placer alr

In [8]:
model.summary()

Model: "my_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 15)             │           195 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 332 (1.30 KB)

 Trainable params: 332 (1.30 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

history = model.fit(
    X_train,y_train,batch_size=1000,
    epochs=40
)

Epoch 1/40
504/504 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7155 - loss: 0.5046
Epoch 2/40
504/504 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8838 - loss: 0.2820
Epoch 3/40
504/504 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8850 - loss: 0.2784
Epoch 4/40
504/504 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8851 - loss: 0.2772
Epoch 5/40
504/504 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8854 - loss: 0.2767
Epoch 6/40
504/504 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8856 - loss: 0.2763
Epoch 7/40
504/504 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8857 - loss: 0.2761
Epoch 8/40
504/504 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8857 - loss: 0.2759
Epoch 9/40
504/504 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8857 - loss: 0.2757
Epoch 10/40
504/504 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8859 - loss: 0.2756
Epoch 11/40
504/504 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8860 - loss: 0.2755
Epoch 12/40
504/504 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

In [10]:

from sklearn.metrics import accuracy_score
y_pred=model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)
accuracy=accuracy_score(y_test,y_pred)
print(accuracy)

3938/3938 ━━━━━━━━━━━━━━━━━━━━ 3s 866us/step
0.8843571428571428


In [11]:
x_test_pca.shape

(270000, 12)

In [12]:
predictions=model.predict(x_test_pca) #invalidargument error comes when test set structure and type does not match with the neural network required structure
predictions=(predictions > 0.5).astype(int).flatten()
submissions=pd.DataFrame({
    'id' : df_test["id"],
    'Heart Disease': predictions
})
submissions.to_csv("submissions.csv", index=False)
submissions.head()


8438/8438 ━━━━━━━━━━━━━━━━━━━━ 7s 852us/step


,id,Heart Disease
0,630000,1
1,630001,0
2,630002,1
3,630003,0
4,630004,0
